In [0]:
%run ../functions/functions

In [0]:
# ==============================
# Lista dos datasets
# ==============================

datasets = [
    "EXP_2021", 
    "EXP_2021_MUN",
    "EXP_2022", 
    "EXP_2022_MUN",
    "IMP_2021", 
    "IMP_2021_MUN",
    "IMP_2022", 
    "IMP_2022_MUN",
    "ISIC_CUCI",
    "NBM",
    "NBM_NCM", 
    "NCM_CGCE",
    "NCM_CUCI",
    "NCM_FAT_AGREG",
    "NCM_ISIC",
    "NCM_PPE",
    "NCM_PPI",
    "NCM_SH",
    "NCM_UNIDADE",
    "PAIS",
    "PAIS_BLOCO",
    "UF",
    "UF_MUN",
    "URF",
    "VIA"
]

In [0]:
dfs = load_all_datasets(datasets)

In [0]:
for nome, df in dfs.items():
    print(nome,',')

In [0]:
#print dos Schemas da bronze
for nome, df in dfs.items():
    print(nome)
    df.printSchema()
    print("-------------------------------------------------------------")

In [0]:
# 1. Configurações específicas para o dataset de Exportação (EXP)
exp_cast_config = {
    "CO_ANO": "int",
    "CO_MES": "int",
    "CO_NCM": "long",
    "CO_UNID": "int",
    "CO_PAIS": "int",
    "CO_VIA": "int",
    "CO_URF": "int",
    "QT_ESTAT": "double",
    "KG_LIQUIDO": "double",
    "VL_FOB": "double"
}

exp_business_keys = [
    "CO_ANO", "CO_MES", "CO_NCM", "CO_UNID", 
    "CO_PAIS", "SG_UF_NCM", "CO_VIA", "CO_URF"
]


In [0]:
# 2. Execução do pipeline
datasets = ["EXP_2021", "EXP_2022"]
dfs = load_all_datasets(datasets)

df_silver_exp = process_silver_layer(
    dfs_dict=dfs, 
    cast_config=exp_cast_config, 
    business_keys=exp_business_keys, 
    sk_name="SK_EXPORTACAO"
)

df_silver_exp.display()

In [0]:
#validacao do union
dfs["EXP_2021"].count() + dfs["EXP_2022"].count() - df_silver_exp.count()

In [0]:
df_silver_exp.printSchema()

In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_exp, 
    table_name="EXP_CONSOLIDADA", 
    primary_keys=["SK_EXPORTACAO"]
)

In [0]:
# Definindo o caminho
silver_path = "abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/EXP_CONSOLIDADA/"

write_silver(df, dataset)

# Lendo os dados
df_silver = spark.read.format("delta").load(silver_path)

# Verificando o resultado
#df_silver.count()

In [0]:
df_silver.count()
#2976163